# Welcome to the AI Agents for Financial Analysis Workshop!

## Part 2: Retrieval-Augmented Generation (RAG) Approach

<a target="_blank" href="https://colab.research.google.com/github/MichaelKarpe/lehman/blob/dev/AI_Agents_for_Financial_Analysis_Part_2.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Welcome to the second part of the workshop! Since you have completed the Full Context approach, you observed how we can pass entire filings into Gemini.

However, passing millions of tokens repeatedly can be slow and expensive. In this notebook, we implement a **RAG pipeline**. We will:
1. Break the SEC filings into smaller "chunks".
2. Convert these chunks into vector embeddings and store them in a Vector Database (Qdrant).
3. Search for the most relevant chunks using semantic search and only feed those into the LLM.

> **❓ Question for you:** Why do we need a vector database like Qdrant for this approach? Why not just use keyword search over a text file?

**Reference SEC Filings (July 2008 10-Q):**
- **PDF**: [https://www.rns-pdf.londonstockexchange.com/rns/8436Z_1-2008-7-24.pdf](https://www.rns-pdf.londonstockexchange.com/rns/8436Z_1-2008-7-24.pdf)
- **Text**: [https://www.sec.gov/Archives/edgar/data/806085/000110465908045115/a08-18147_110q.htm](https://www.sec.gov/Archives/edgar/data/806085/000110465908045115/a08-18147_110q.htm)

### 🔑 API Setup
1. **Create API key** at the following link: [https://aistudio.google.com/u/1/api-keys](https://aistudio.google.com/u/1/api-keys)
2. When the notebook is opened in Google Colab, click on **"Secrets"** on the left panel, then **"Gemini API keys"** and **"Import key from Google AI Studio"** in order to get your Gemini key API. Toggle the **"notebook access"** button to activate it if not done automatically.

In [1]:
!pip install -q edgartools==5.21.1 langchain==1.2.10 langchain-google-genai==4.2.1 langchain-qdrant==1.1.0 langchain-text-splitters==1.1.1 qdrant-client==1.17.0 pydantic==2.12.3 rich==13.9.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.4/390.4 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 9.9 MB/s eta 0:00:00


In [5]:
import os
import time
from typing import List, Optional
from datetime import datetime
from pydantic import BaseModel, Field
from edgar import Company, set_identity
from google import genai
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from qdrant_client import QdrantClient
from qdrant_client.http import models
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich import box
from rich.text import Text

# Constants
LEHMAN_CIK = "806085"
set_identity("your_email@example.com")

# Setup Google API Key
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")

console = Console()
console.print("✓ Libraries imported and environment configured.")

SecretNotFoundError: Secret GEMINI_API_KEY does not exist.

### Step 1: Fetching SEC Data for the RAG Pipeline

We will use `edgartools` again to retrieve the 10-K and 10-Q filings for Lehman Brothers. For the RAG pipeline, we extract the pure text content from the filings.


In [3]:
def fetch_lehman_filings():
    lehman = Company(LEHMAN_CIK)
    forms = ["10-K", "10-Q"]
    target_filings = []

    start_date = "2008-07-01"

    for form in forms:
        filings = lehman.get_filings(form=form)
        for f in filings:
            # f.filing_date is usually a string like 'YYYY-MM-DD' or a date object
            f_date_str = str(f.filing_date)
            if f_date_str >= start_date:
                target_filings.append(f)

    return target_filings

filings = fetch_lehman_filings()
console.print(f"✓ Found {len(filings)} filings from July 2008 onwards to analyze.")
for f in filings:
    console.print(f"  • {f.form} filed on {f.filing_date}")

✓ Found 1 filings from July 2008 onwards to analyze.

• 10-Q filed on 2008-07-10

### Step 2: Chunking Text and Creating Vector Embeddings

**📊 Monitor Quotas:**
Monitor Gemini embedding and LLM models quotas when running the cells calling the models at the following links:
- [https://aistudio.google.com/u/1/rate-limit](https://aistudio.google.com/u/1/rate-limit)
- [https://console.cloud.google.com/apis/api/generativelanguage.googleapis.com/quotas](https://console.cloud.google.com/apis/api/generativelanguage.googleapis.com/quotas)
*(On the second link, sort by descending order of "Current usage percentage" to see your current usage)*

This is the core of RAG. We split the long SEC filings into smaller pieces (chunks) of text. Then, we use an Embedding Model (`gemini-embedding-001`) to convert each chunk into a mathematical vector and store it in Qdrant.

> **❓ Question for you:** How does the chunk size and chunk overlap affect the quality of the retrieved context? What happens to the LLM's understanding if the chunks are too small or too large?



In [6]:
def prepare_vector_store(filings):
    if not filings:
        return None

    all_chunks = []
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=8192,
        chunk_overlap=1024
    )

    for f in filings:
        console.print(f"Extracting text from {f.form} ({f.filing_date})...")
        try:
            text = f.text()
            chunks = text_splitter.create_documents(
                texts=[text],
                metadatas=[{"form": f.form, "date": str(f.filing_date), "accession": f.accession_no}]
            )
            all_chunks.extend(chunks)
        except Exception as e:
            console.print(f"[red]Error extracting text from {f.accession_no}: {e}[/red]")

    embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

    # Initialize in-memory Qdrant
    client = QdrantClient(":memory:")
    client.create_collection(
        collection_name="lehman_filings",
        vectors_config=models.VectorParams(size=3072, distance=models.Distance.COSINE),
    )

    vector_store = QdrantVectorStore(
        client=client,
        collection_name="lehman_filings",
        embedding=embeddings,
    )

    # Precise Rate Limiting Logic
    TPM_LIMIT = 30000
    RPM_LIMIT = 100

    tokens_sent_in_minute = 0
    requests_sent_in_minute = 0
    window_start = time.time()

    console.print(f"✓ Indexing {len(all_chunks)} chunks with a strict 30k TPM limit...")

    # Initialize Gemini client for precise token counting
    client = genai.Client()

    for i, chunk in enumerate(all_chunks):
        # Precise token count using Gemini SDK
        chunk_tokens = client.models.count_tokens(model="gemini-3-flash-preview", contents=chunk.page_content).total_tokens

        # Check if we would exceed either limit (TPM or RPM)
        if (tokens_sent_in_minute + chunk_tokens > TPM_LIMIT) or (requests_sent_in_minute + 1 > RPM_LIMIT):
            # Calculate precise wait time until the minute resets
            elapsed = time.time() - window_start
            wait_time = max(60 - elapsed, 0) + 1 # +1 for safety margin

            console.print(f"  [yellow]Limit reached ({tokens_sent_in_minute} tokens). Waiting {wait_time:.1f}s for reset...[/yellow]")
            time.sleep(wait_time)

            # Reset window
            window_start = time.time()
            tokens_sent_in_minute = 0
            requests_sent_in_minute = 0

        # Add the document
        vector_store.add_documents([chunk])

        # Update counters
        tokens_sent_in_minute += chunk_tokens
        requests_sent_in_minute += 1

        if (i + 1) % 5 == 0:
            console.print(f"  Indexed {i+1}/{len(all_chunks)} chunks... ({tokens_sent_in_minute} tokens in current window)")

    return vector_store

vector_store = prepare_vector_store(filings)
if vector_store:
    console.print(f"✓ Vector store initialized and indexed.")

Extracting text from 10-Q (2008-07-10)...

✓ Indexing 77 chunks with a strict 30k TPM limit...

Indexed 5/77 chunks... (6427 tokens in current window)

Indexed 10/77 chunks... (14954 tokens in current window)

Indexed 15/77 chunks... (22422 tokens in current window)

...

Indexed 65/77 chunks... (12521 tokens in current window)

Indexed 70/77 chunks... (21169 tokens in current window)

Indexed 75/77 chunks... (29430 tokens in current window)

Limit reached (29430 tokens). Waiting 54.6s for reset...

✓ Vector store initialized and indexed.

### Step 3: Define the QA Chain with Exact Citations

We will configure `pydantic` models to retrieve the relevant chunks from Qdrant and answer our prompt. Crucially, we require the model to provide **exact quotes** (citations) from the retrieved text to justify its identified warning signs.

> **❓ Question for you:** How does asking the agent to provide exact quotes from the retrieved chunks mitigate the risk of LLM "hallucinations" in an auditing context?


In [ ]:
class BankruptcyWarningSign(BaseModel):
    warning_type: str = Field(description="Type of warning (e.g., Leverage, Exposure, Capital, Liquidity)")
    key_metric: str = Field(description="Specific financial metric or event")
    value: str = Field(description="Quantified value if available")
    severity: int = Field(description="Severity score 1-10", ge=1, le=10)
    description: str = Field(description="Detailed explanation of the concern")
    exact_passage: str = Field(description="Supporting quote from the filing")

class InvestigationReport(BaseModel):
    findings: List[BankruptcyWarningSign]
    summary: str


# Change the model in case of quota exceeded
# model = "gemini-2.5-flash"
model = "gemini-3-flash-preview"

llm = ChatGoogleGenerativeAI(model=model, temperature=0)
structured_llm = llm.with_structured_output(InvestigationReport)

def run_investigation(vector_store):
    if not vector_store:
        return None

    queries = [
        "Analyze leverage ratios and debt levels. Look for high debt-to-equity or net leverage ratios.",
        "Investigate mortgage-related exposure, specifically subprime and commercial mortgage-backed securities.",
        "Search for mentions of capital raises, liquidity concerns, or funding difficulties.",
        "Identify significant losses in quarterly earnings, especially in the capital markets segment.",
        "Look for Repo 105 or other off-balance sheet accounting treatments used to manage leverage."
    ]

    all_context = []
    for query in queries:
        docs = vector_store.similarity_search(query, k=5)
        all_context.extend([d.page_content for d in docs])

    combined_context = "\n---\n".join(list(set(all_context))[:10])

    prompt = ChatPromptTemplate.from_template("""
    You are a financial forensic analyst. Based ONLY on the following context from Lehman Brothers' 2008 filings,
    identify specific bankruptcy warning signs.

    Context:
    {context}

    Focus on numbers, specific disclosures, and risk factors that preceded the bankruptcy.
    """)

    chain = prompt | structured_llm
    report = chain.invoke({"context": combined_context})
    return report

report = run_investigation(vector_store)

### Step 4: Display the RAG Report

We display the formatted report. Because we used RAG with exact citations, you can now copy any extracted quote and search for it directly within the original [Lehman 10-Q July 2008 (SEC Text Version)](https://www.sec.gov/Archives/edgar/data/806085/000110465908045115/a08-18147_110q.htm) to verify its accuracy and context.

> **🔍 Verification Task:**
> Please look for the **"supporting passages"** displayed as the output of this cell in the PDF or text formats of the filings (links provided at the beginning of the notebook). Make sure that these exact passages are taken from the original documents and are not hallucinated!



In [ ]:
def display_report(report):
    if not report:
        console.print("[red]No report generated. check vector store initialization.[/red]")
        return

    console.print(Panel(Text("LEHMAN BROTHERS BANKRUPTCY INVESTIGATION REPORT", style="bold white on red"), box=box.DOUBLE))

    table = Table(title="Detected Warning Signs", box=box.ROUNDED)
    table.add_column("Type", style="cyan")
    table.add_column("Metric", style="magenta")
    table.add_column("Value", style="green")
    table.add_column("Sev", justify="center")
    table.add_column("Description", style="white")

    for finding in report.findings:
        sev_color = "red" if finding.severity >= 8 else "yellow" if finding.severity >= 5 else "green"
        table.add_row(
            finding.warning_type,
            finding.key_metric,
            finding.value,
            f"[{sev_color}]{finding.severity}[/{sev_color}]",
            finding.description
        )

    console.print(table)
    console.print(Panel(report.summary, title="Executive Summary", border_style="blue"))

    console.print("\n[bold]Supporting Passages:[/bold]")
    for i, finding in enumerate(report.findings):
        console.print(f"\n{i+1}. [italic]'{finding.exact_passage}'[/italic]")

display_report(report)